# BERT LLM Hallucination Detector

### Imports

In [17]:
import os
import json
import random
import ast
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from dotenv import load_dotenv
from tavily import TavilyClient
from langdetect import detect, DetectorFactory
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

### Step 1: Initialization and Setup

In [3]:
load_dotenv()
API_Tavily = os.getenv("API_Tavily")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

DetectorFactory.seed = 0

# Initialize clients and tokenizer
client = TavilyClient(API_Tavily)
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

# Initialize Translation/Groq LLM
model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
)

### Step 2: Data Loading

In [4]:
dataset = load_dataset("Helsinki-NLP/mu-shroom", "all")

# Access splits
train = dataset["train_unlabeled"]
val = dataset["validation"]
test = dataset["test"]

### Step 3: Translation and Normalization Logic

In [5]:
def detect_language(text):
    """
    Detect language using ISO-639-1 codes:
    en, de, fr, hi, etc.
    """
    try:
        return detect(text)
    except Exception:
        return "unknown"

def translate_text(text, target_language):
    messages = [
    ("system", f"You are a helpful translator. Translate the user sentence to {target_language}."),
    ("human", text),
    ]
    response = model.invoke(messages)
    return response.content.strip()

def normalize_answer(question, answer):
    """
    Check if question and answer languages match.
    Translate answer if needed.
    """
    question_lang = detect_language(question)
    answer_lang = detect_language(answer)

    if question_lang == answer_lang:
        return answer

    return translate_text(
        answer,
        question_lang
    )


### Step 4: Ground Truth Retrieval (Tavily)

In [6]:
unique_questions = list(set(train["model_input"]))

# Get reference first old else new
if os.path.exists("reference_map.json"):
    with open("reference_map.json", "r", encoding="utf-8") as f:
        reference_map = json.load(f)
else:
    reference_map = {}

# Search questions
for question in unique_questions:
    # Skip already searched questions
    if question in reference_map:
        continue

    # Get answers
    try:
        response = client.search(
            query=question,
            include_answer="basic",
            search_depth="basic"
        )
        answer = response.get("answer") or "No answer returned"
        answer = normalize_answer(question, answer)
        reference_map[question] = answer
        
        # Save immediately after each successful search
        with open("reference_map.json", "w", encoding="utf-8") as f:
            json.dump(reference_map, f, indent=4, ensure_ascii=False)
            
    except Exception as e:
        continue


### Step 5: Map References to Dataset

In [7]:
def add_reference(example):
    example["reference_answer"] = reference_map[example["model_input"]]
    return example

train = train.map(add_reference)

Map:   0%|          | 0/3351 [00:00<?, ? examples/s]

In [ ]:
train['model_input']

Column(['Do all arthropods have antennae?', 'Do all arthropods have antennae?', 'Do all arthropods have antennae?', 'Do all arthropods have antennae?', 'Do all arthropods have antennae?', ...])

### Step 6: LLM Hallucination Detection Prompting

#### 6.1 Prompt Templates

In [ ]:
def prompt_template(model_input,model_output_text,reference_answer):
    """
    Generate 5 prompts for hallucination detection.
    params: model_input, model_output_text, reference_answer
    return: list of 5 prompts
    """
    # default
    message1 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - hallucinated_span must be copied EXACTLY from the LLM answer.
    - Do not paraphrase or correct the text.
    - Include only false or unsupported factual claims.
    - Include complete factual claims that are false or unsupported.
    - The spans must be as short as possible, but include the complete factual claim that is false.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Atomic claim decomposition
    message2 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Identify each individual false factual claim separately.
    - If a sentence contains multiple hallucinated facts, return separate spans.
    - Do not include correct parts of the sentence.
    - Do not mark equivalent statements as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Strict contradiction detector
    message3 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Only return claims that directly contradict the reference answer.
    - Do not mark missing information as hallucination.
    - Do not mark different wording with the same meaning as hallucination.
    - Avoid guessing.
    - If no direct contradiction exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Complete factual claim
    message4 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return complete factual claims that are false or unsupported.
    - Include enough context to understand why the claim is wrong.
    - Do not return isolated words or numbers.
    - Do not mark correct paraphrases as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # smallest hall
    message5 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return only statements that are factually incorrect or unsupported.
    - Select the smallest span that contains the hallucinated information.
    - Do not include surrounding correct information.
    - Do not mark paraphrases or equivalent meanings as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Context-aware balanced annotation
    message6 = [(
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return the shortest span that still represents a complete false factual claim.
    - Consider the meaning of the statement, not only exact wording.
    - Do not mark correct explanations, paraphrases, or logical consequences of the reference answer.
    - Do not include unsupported speculation unless it conflicts with the reference answer.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]

    messages = [message1,message2,message3,message4,message5,message6]

    return messages

#### 6.2 Model setup

In [14]:
def create_llm_models():

    models = {

        "qwen3.6-27b": ChatGroq(
            model="qwen/qwen3.6-27b",
            temperature=0,
            max_tokens=None,
            reasoning_effort="none",
            reasoning_format="hidden",
            timeout=None,
            max_retries=2,
        ),

        "llama-3.3-70b": ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
        ),

        "gpt-oss-120b": ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0,
            max_tokens=None,
            reasoning_effort="low",
            reasoning_format="hidden",
            timeout=None,
            max_retries=2,
        )
    }

    return models

Helper functions for LLM hallucination detection

In [ ]:
def parse_spans(output):
    """
    Convert LLM response into list of hallucinated spans.
    """

    try:
        spans = ast.literal_eval(output.strip())
    except Exception:
        return []

    cleaned = []

    for span in spans:

        # if model returns tuple accidentally
        if isinstance(span, (tuple, list)):
            span = span[0]

        if isinstance(span, str):
            cleaned.append(span)

    return cleaned

def spans_to_character_labels(text, spans):
    """
    Convert list of hallucinated spans into character-level labels.
    """
    labels = np.zeros(len(text), dtype=np.int8)

    for span in spans:

        start = text.find(span)

        if start == -1:
            continue

        end = start + len(span)

        labels[start:end] = 1

    return labels

In [23]:
cl =parse_spans(
    '["The capital of France is Berlin.", " The Eiffel Tower is located in New York City."]'
)
span_labels = spans_to_character_labels(
    "I like The capital of France is Berlin. The Eiffel Tower is located in New York City.", cl)
print(span_labels)

[0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1]


#### 6.3 Annotation

In [16]:
def annotate_sample(sample, llm_models):

    text = sample["model_output_text"]

    prompts = prompt_template(
        sample["model_input"],
        sample["model_output_text"],
        sample["reference_answer"]
    )

    all_character_labels = []

    metadata = []


    for model_name, llm in llm_models.items():

        for prompt_id, prompt in enumerate(prompts):

            response = llm.invoke(prompt)

            spans = parse_spans(response.content)

            labels = spans_to_character_labels(
                text,
                spans
            )

            all_character_labels.append(labels)


            metadata.append({
                "model": model_name,
                "prompt": prompt_id,
                "spans": spans
            })


    # shape:
    # (18, number_of_characters)
    label_matrix = np.stack(all_character_labels)


    soft_labels = label_matrix.mean(axis=0)

    hard_labels = (
        soft_labels >= 0.5
    ).astype(np.int8)


    return {
        "text": text,
        "soft_labels": soft_labels.tolist(),
        "hard_labels": hard_labels.tolist(),
        "metadata": metadata
    }

### Annotate Dataset

In [ ]:
llm_models = create_llm_models()

annotations = []

for i, sample in enumerate(train):

    print("Processing:", i)

    result = annotate_sample(
        sample,
        llm_models
    )

    annotations.append(result)

In [ ]:
# Invoke Model
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    max_tokens=None,
    reasoning_effort="none",
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

for i, message in enumerate(messages):
    ai_msg = llm.invoke(message)

# ==========================================
# Step 7: Parse Word Spans to Indexes
# ==========================================
output = ast.literal_eval(ai_msg.content)
text = sample["model_output_text"]
results = []

for span in output:
    start = text.find(span)

    if start == -1:
        continue

    end = start + len(span) - 1

    results.append({
        "span": span,
        "char_start": start,
        "char_end": end,
        "label": 1
    })

# ==========================================
# Step 8: Tokenization & Offset Mapping
# ==========================================
# Process text into specific token IDs and offsets based on extracted spans
example = dataset["validation"][100]  # Example reference extraction
text = example["model_output_text"]
char_spans = example["hard_labels"]  # list of [start,end] spans

enc = tokenizer(text, return_offsets_mapping=True)
offsets = enc.offset_mapping

# Initialize label for every word as -100
labels = [-100] * len(offsets)

for (span_start, span_end) in char_spans:
    for i, (s, e) in enumerate(offsets):
        if e <= span_start or s >= span_end:
            continue
        # label first sub-token of each word overlapping a span
        if s == 0 and e != 0:
            labels[i] = 1